In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
import ROOT
from analysis_framework import Dataset
from OptimalObservableHelper import OptimalObservableHelper
from AltSetupHandler import AltSetupHandler

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x7b2a6c0
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x7d17900


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
# n_threads = 12
n_threads = 10
# n_threads = 6
no_rvec = True
# write_outputs = False
write_outputs = True
# dataset_path = "data/datasets/reweighted/signal-only.json"
dataset_path = "data/datasets/truth-reweighted-hel-mW-new2/signal-only.json"
# friend_dataset_path = "data/datasets/truth-reweighted/signal-only.json"
# friend_dataset_path = "data/datasets/truth-reweighted-hel/signal-only.json"
friend_paths = [
    # "data/datasets/truth-reweighted-hel-mW/signal-only.json",
    "data/datasets/oo-sqme-new2/signal-only.json",
    "data/datasets/oo-sqme-new2/signal-only-kinfit.json",
    "data/datasets/oo-sqme-new2/signal-only-extras.json",
    "data/datasets/oo-sqme-new2/signal-only-cc10.json",
]

plot_path = "plots/oo-val-new2-polfix"
# plot_oo_name = "kinfit_clean_reco_oo"
# plot_oo_name = "mlvec_clean_reco_oo"
# plot_oo_name = "nomb_mc_oo"

# out_dir = "fit-configs/signal-only-mW-pol-new2"
out_dir = "fit-configs/signal-only-mW-pol-new2-polfix"

pol_configs = [
    (0., 0.),
    (-0.8, -0.6),
    (-0.8, 0.6),
    (0.8, -0.6),
    (0.8, 0.6),
    (-0.8, -0.3),
    (-0.8, 0.3),
    (0.8, -0.3),
    (0.8, 0.3),
    (0.8, 0.0),
    (-0.8, 0.0),
]

pars = ["g1z", "ka", "la", "mW", "epol", "ppol"]

In [4]:
ROOT.EnableImplicitMT(n_threads)
# environ["OMP_NUM_THREADS"] = "6"

In [5]:
oo_calc_order = 8
# oo_calc_order = 5

oo_configs = [
    f"g1z_pos_1em{oo_calc_order:02}",
    f"ka_pos_1em{oo_calc_order:02}",
    f"la_pos_1em{oo_calc_order:02}",
    f"mW_pos_1em{oo_calc_order:02}",
    ]

In [6]:
# alt_setup_handler = AltSetupHandler("""
# {
#   "SM": {
#     "mW": 80.419,
#     "g1z": 1.0,
#     "ka": 1.0,
#     "la": 0.0
#   },
# "variations": [
#     0.1,
#     -0.1,
#     0.01,
#     -0.01,
#     0.002,
#     -0.002,
#     0.001,
#     -0.001,
#     7.5e-04,
#     -7.5e-04,
#     5e-04,
#     -5e-04,
#     2.5e-04,
#     -2.5e-04,
#     1e-04,
#     -1e-04,
#     1e-05,
#     1e-06,
#     1e-07,
#     1e-08
#   ]
# }
# """, mirror=False, combinations=False)
alt_setup_handler = AltSetupHandler("""
{
  "SM": {
    "mW": 80.419,
    "g1z": 1.0,
    "ka": 1.0,
   "la": 0.0
  },
"variations": [
    1e-08
  ]
}
""", mirror=False, combinations=False)
alt_config_names = list(alt_setup_handler.get_alt_setup().keys())

In [7]:
for e_pol, p_pol in pol_configs:
    dataset = Dataset.from_json(dataset_path)
    friend_datasets = [Dataset.from_json(friend_path) for friend_path in friend_paths]
    analysis = OptimalObservableHelper(dataset, friend_datasets=friend_datasets)
    analysis.init_categories()
    # check if we missed any processes
    # print(analysis.is_complete_categorisation())
    signal_category = ["4f_sw_sl_signal"]
    oo_names = {
        "reco_oo": analysis.define_optimal_observables_polarised("O", ["reco_sqme_hels", "wj_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "mlvec_reco_oo": analysis.define_optimal_observables_polarised("mlvec_O", ["mlvec_reco_sqme_hels", "wj_mlvec_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "reco_jm_oo": analysis.define_optimal_observables_polarised("jm_O", ["reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "mlvec_reco_jm_oo": analysis.define_optimal_observables_polarised("mlvec_jm_O", ["mlvec_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "clean_reco_oo": analysis.define_optimal_observables_polarised("clean_O", ["clean_reco_sqme_hels", "wj_clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "clean_brems_reco_oo": analysis.define_optimal_observables_polarised("clean_brems_O", ["clean_brems_reco_sqme_hels", "wj_clean_brems_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "kinfit_clean_reco_oo": analysis.define_optimal_observables_polarised("kinfit_clean_O", ["kinfit_clean_reco_sqme_hels", "wj_kinfit_clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "mlvec_clean_reco_oo": analysis.define_optimal_observables_polarised("mlvec_clean_O", ["mlvec_clean_reco_sqme_hels", "wj_mlvec_clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "mlvec_clean_brems_reco_oo": analysis.define_optimal_observables_polarised("mlvec_clean_brems_O", ["mlvec_clean_brems_reco_sqme_hels", "wj_mlvec_clean_brems_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "clean_reco_jm_oo": analysis.define_optimal_observables_polarised("clean_jm_O", ["clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "clean_brems_reco_jm_oo": analysis.define_optimal_observables_polarised("clean_brems_jm_O", ["clean_brems_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "kinfit_clean_reco_jm_oo": analysis.define_optimal_observables_polarised("kinfit_clean_jm_O", ["kinfit_clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "mlvec_clean_reco_jm_oo": analysis.define_optimal_observables_polarised("mlvec_clean_jm_O", ["mlvec_clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "mlvec_clean_brems_reco_jm_oo": analysis.define_optimal_observables_polarised("mlvec_clean_brems_jm_O", ["mlvec_clean_brems_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "cheat_clean_reco_oo": analysis.define_optimal_observables_polarised("cheat_clean_O", ["cheat_clean_reco_sqme_hels", "wj_cheat_clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "mlvec_cheat_clean_reco_oo": analysis.define_optimal_observables_polarised("mlvec_cheat_clean_O", ["mlvec_cheat_clean_reco_sqme_hels", "wj_mlvec_cheat_clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "cheat_clean_reco_jm_oo": analysis.define_optimal_observables_polarised("cheat_clean_jm_O", ["cheat_clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "mlvec_cheat_clean_reco_jm_oo": analysis.define_optimal_observables_polarised("mlvec_cheat_clean_jm_O", ["mlvec_cheat_clean_reco_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "mc_oo": analysis.define_optimal_observables_polarised("mc_O", ["mc_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nomb_mc_oo": analysis.define_optimal_observables_polarised("nomb_mc_O", ["nomb_mc_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nomb_mc_rlep_oo": analysis.define_optimal_observables_polarised("nomb_mc_rlep_O", ["nomb_mc_rlep_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nomb_mc_rlep_gamma_oo": analysis.define_optimal_observables_polarised("nomb_mc_rlep_gamma_O", ["nomb_mc_rlep_gamma_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nomb_mc_rlep_brems_oo": analysis.define_optimal_observables_polarised("nomb_mc_rlep_brems_O", ["nomb_mc_rlep_brems_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nomb_mc_rlep_cheated_brems_oo": analysis.define_optimal_observables_polarised("nomb_mc_rlep_cheated_brems_O", ["nomb_mc_cheated_brems_rlep_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nurec_nomb_mc_oo": analysis.define_optimal_observables_polarised("nurec_nomb_mc_O", ["nurec_nomb_mc_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nurec_nomb_mc_rlep_oo": analysis.define_optimal_observables_polarised("nurec_nomb_mc_rlep_O", ["nurec_nomb_mc_rlep_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nurec_nomb_mc_rlep_gamma_oo": analysis.define_optimal_observables_polarised("nurec_nomb_mc_rlep_gamma_O", ["nurec_nomb_mc_rlep_gamma_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nurec_nomb_mc_rlep_brems_oo": analysis.define_optimal_observables_polarised("nurec_nomb_mc_rlep_brems_O", ["nurec_nomb_mc_rlep_brems_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nurec_nomb_mc_rlep_cheated_brems_oo": analysis.define_optimal_observables_polarised("nurec_nomb_mc_rlep_cheated_brems_O", ["nurec_nomb_mc_rlep_cheated_brems_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_mc_oo": analysis.define_optimal_observables_polarised("av_mc_O", ["mc_sqme_hels", "wj_mc_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nomb_mc_oo": analysis.define_optimal_observables_polarised("av_nomb_mc_O", ["nomb_mc_sqme_hels", "wj_nomb_mc_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nomb_mc_rlep_oo": analysis.define_optimal_observables_polarised("av_nomb_mc_rlep_O", ["nomb_mc_rlep_sqme_hels", "wj_nomb_mc_rlep_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nomb_mc_rlep_gamma_oo": analysis.define_optimal_observables_polarised("av_nomb_mc_rlep_gamma_O", ["nomb_mc_rlep_gamma_sqme_hels", "wj_nomb_mc_rlep_gamma_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nomb_mc_rlep_brems_oo": analysis.define_optimal_observables_polarised("av_nomb_mc_rlep_brems_O", ["nomb_mc_rlep_brems_sqme_hels", "wj_nomb_mc_rlep_brems_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nomb_mc_rlep_cheated_brems_oo": analysis.define_optimal_observables_polarised("av_nomb_mc_rlep_cheated_brems_O", ["nomb_mc_cheated_brems_rlep_sqme_hels", "wj_nomb_mc_cheated_brems_rlep_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nurec_nomb_mc_oo": analysis.define_optimal_observables_polarised("av_nurec_nomb_mc_O", ["nurec_nomb_mc_sqme_hels", "wj_nurec_nomb_mc_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nurec_nomb_mc_rlep_oo": analysis.define_optimal_observables_polarised("av_nurec_nomb_mc_rlep_O", ["nurec_nomb_mc_rlep_sqme_hels", "wj_nurec_nomb_mc_rlep_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nurec_nomb_mc_rlep_gamma_oo": analysis.define_optimal_observables_polarised("av_nurec_nomb_mc_rlep_gamma_O", ["nurec_nomb_mc_rlep_gamma_sqme_hels", "wj_nurec_nomb_mc_rlep_gamma_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nurec_nomb_mc_rlep_brems_oo": analysis.define_optimal_observables_polarised("av_nurec_nomb_mc_rlep_brems_O", ["nurec_nomb_mc_rlep_brems_sqme_hels", "wj_nurec_nomb_mc_rlep_brems_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nurec_nomb_mc_rlep_cheated_brems_oo": analysis.define_optimal_observables_polarised("av_nurec_nomb_mc_rlep_cheated_brems_O", ["nurec_nomb_mc_rlep_cheated_brems_sqme_hels", "wj_nurec_nomb_mc_rlep_cheated_brems_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "nurec_post94_mc_oo": analysis.define_optimal_observables_polarised("nurec_post94_mc_O", ["nurec_post94_mc_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "av_nurec_post94_mc_oo": analysis.define_optimal_observables_polarised("av_nurec_post94_mc_O", ["nurec_post94_mc_sqme_hels", "wj_nurec_post94_mc_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "av_nurec_nomb_mc_cc10_oo": analysis.define_optimal_observables_polarised("av_nurec_nomb_mc_cc10_O", ["nurec_nomb_mc_cc10_sqme_hels", "wj_nurec_nomb_mc_cc10_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        "mlvec_clean_brems_reco_cc10_oo": analysis.define_optimal_observables_polarised("mlvec_clean_brems_cc10_O", ["mlvec_clean_brems_reco_cc10_sqme_hels", "wj_mlvec_clean_brems_reco_cc10_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
        # "mlvec_clean_brems_reco_san_oo": analysis.define_optimal_observables_polarised("mlvec_clean_brems_san_O", ["mlvec_clean_brems_reco_san_sqme_hels", "wj_mlvec_clean_brems_reco_san_sqme_hels"], oo_configs, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
    }
    analysis.add_filter("&&".join([f"std::isfinite({oo})" for oo_name in oo_names.values() for oo in oo_name]), "finite OO")
    analysis.book_reports()
    weight_names = analysis.book_weight_sums(["nominal"] + alt_config_names, categories=signal_category, hel=True)
    for names in oo_names.values():
        analysis.define_weighted_oo(names, weight_names, categories=signal_category)
        analysis.book_oo_sums(names, weight_names, categories=signal_category)
        analysis.book_oo_matrix(names, categories=signal_category)
    analysis.run()
    analysis.print_reports()
    if write_outputs:
        for name, names in oo_names.items():
            # print(name)
            analysis.print_fit_input(names, weight_names, pars, e_pol=e_pol, p_pol=p_pol, categories=signal_category, dir=out_dir, name=name, hel=True)
    del analysis
    del dataset
    del friend_datasets

         4f_sw_sl_signal
        10041022 (1e-03) All
        10040472 (1e-03) finite OO
                    1.00 efficiency
                    1.00 purity

# pars, evt/ab_inv, means, slopes, cov
['g1z', 'ka', 'la', 'mW', 'epol', 'ppol']
2008094.3195957248
[-0.04489855221587967, -0.03408330480554171, 0.010267555346887393, 0.19772982051183316, -0.9310741278193931, 0.9293029202276837]
[[-11.640555627636491, -2.2676387706419137, 7.6511929376421355, -0.14273933467877584, -0.04648873462731547, 0.0555117203119999], [-3.243805191084959, -6.44213655230202, 1.763335819980299, -0.08119009413347524, -0.08572033718970079, 0.047455009583942824], [-39.914548111728934, -4.757632162978926, 79.85102343237789, 0.6155381038538129, 0.5372788182002575, -0.2392495757463837], [0.043148483913465166, 0.004935831005021665, -0.015934954632982105, 0.23370900894067143, -0.013307205443812743, 0.013488003571785506], [0.0028378189204908717, -0.0003289742686559211, -0.01222410860419136, 0.005200104337266562, -0.01639